# Week 5 — Retrieval 고도화 (Retrieval Ablation)

## 목표
4주차에서 채택한 chunk 전략 위에서, **dense 단일 검색의 한계를 넘어 검색 품질을 엔지니어링적으로 개선**한다.
같은 golden set으로 retrieval 구성을 단계별로 바꿔가며 RAGAS로 측정하고, baseline 대비 개선 폭을 수치화한다.

선행: `week4_chunking_experiments.ipynb` (전략 C 채택), `week4_retrospective.md`

## Ablation 구성
| config | 구성 | 격리하는 효과 |
|---|---|---|
| `dense_only` | Dense(e5-base) top-k (= 4주차 전략 C 결과) | baseline |
| `bm25_ws` | BM25 sparse, 공백 토크나이저 | sparse 단독 + 토크나이저 baseline |
| `bm25_kiwi` | BM25 sparse, Kiwi 형태소 토크나이저 | **한국어 토크나이저 효과** (vs bm25_ws) |
| `hybrid` | Dense + BM25(Kiwi), RRF 앙상블 | sparse+dense 결합 효과 |
| `hybrid_rerank` | hybrid 후보 → BGE-reranker 재정렬 | re-ranking 효과 |
| `hybrid_rerank_mq` | Multi-Query로 질의 확장 → hybrid+rerank | query 변환 효과 |

## 설계 원칙
- embedding/LLM/chunk는 4주차와 동일 고정 → **retrieval 구성만** 변수
- 한 단계에 한 가지만 추가해서 각 기법의 기여를 분리 측정
- RAGAS 3지표 + **config별 평균 latency** 기록 (rerank/MQ는 느려짐 → ADR 근거)


---
## 0. 패키지 (4주차 + reranker/kiwi 추가)

In [ ]:
# 4주차 환경에 아래만 추가로 필요
# %pip install -q rank_bm25 kiwipiepy sentence-transformers
# (sentence-transformers는 3주차에 이미 설치됨 / BGE reranker가 이걸로 동작)

---
## 1. 경로 / 설정 (CONFIG)

baseline(embedding/LLM/chunk)은 4주차와 동일 고정. retrieval 파라미터만 조정한다.

In [ ]:
from pathlib import Path
import os, json, time

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_EVAL = PROJECT_ROOT / "data" / "eval"
VECTOR_ROOT = PROJECT_ROOT / "data" / "vector_store"

# ---- 4주차와 동일 고정 ----
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
OLLAMA_MODEL = "qwen3:4b"
EMBED_DEVICE = "cpu"            # GPU 있으면 "cuda" (reranker도 빨라짐)

# 5주차가 올라탈 4주차 chunk 전략 (week4_{이름} 컬렉션이 있어야 함)
CHOSEN_CHUNK_STRATEGY = "C_cleaned"

# ---- retrieval 파라미터 ----
TOP_K = 5             # 최종 컨텍스트 개수 (baseline과 동일)
RRF_FETCH = 10        # hybrid에서 각 retriever가 가져올 후보 수
RERANK_FETCH = 20     # rerank 전 후보 pool 크기
MULTI_QUERY_N = 3     # 원 질문 외 생성할 변형 질의 수
RRF_K = 60            # RRF 상수
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"   # 다국어(한/영) cross-encoder

# 돌릴 config 선택 (CPU면 시간 큼 -> 일부만 켜고 단계적으로)
RUN_CONFIGS = ["dense_only", "bm25_ws", "bm25_kiwi", "hybrid", "hybrid_rerank", "hybrid_rerank_mq"]

# dense_only는 4주차 전략 C RAGAS 결과가 있으면 재사용
REUSE_WEEK4_DENSE = True

print("chunk 전략:", CHOSEN_CHUNK_STRATEGY)
print("run configs:", RUN_CONFIGS)

---
## 2. 4주차 chunk 로드 (Chroma 컬렉션에서 그대로 복원)

dense 인덱스를 재생성하지 않는다. 4주차에서 만든 컬렉션을 열고, **동일한 chunk**를 꺼내 BM25에도 그대로 사용 → dense/sparse가 같은 chunk를 보게 만들어 비교를 공정하게 한다.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
import chromadb

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": EMBED_DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)

vdir = VECTOR_ROOT / f"week4_{CHOSEN_CHUNK_STRATEGY}"
coll = f"breast_rag_week4_{CHOSEN_CHUNK_STRATEGY}"
client = chromadb.PersistentClient(path=str(vdir))
assert coll in [c.name for c in client.list_collections()], \
    f"{coll} 컬렉션 없음 -> 먼저 week4_chunking_experiments.ipynb로 전략 {CHOSEN_CHUNK_STRATEGY} 인덱싱 필요"

vectorstore = Chroma(collection_name=coll, embedding_function=embeddings, persist_directory=str(vdir))
print(f"dense 컬렉션 로드: {vectorstore._collection.count()}개 chunk")

# 동일 chunk 복원 (BM25용)
raw = vectorstore.get(include=["documents", "metadatas"])
chunk_docs = [Document(page_content=t, metadata=m or {})
              for t, m in zip(raw["documents"], raw["metadatas"])]
print(f"BM25용 chunk 복원: {len(chunk_docs)}개")

---
## 3. Retriever 구성 요소

전부 `List[Document]`를 반환하는 함수로 만들어 조합을 명시적으로 통제한다 (포트폴리오에서 RRF/rerank 로직을 직접 보여주기 위함).

In [ ]:
# (1) Dense
def dense_search(query: str, k: int):
    return vectorstore.similarity_search(query, k=k)

# (2) BM25 - 토크나이저 2종
from langchain_community.retrievers import BM25Retriever

def ws_tokenize(text: str):
    return text.split()

# Kiwi 형태소 토크나이저 (명사/어간 중심)
from kiwipiepy import Kiwi
_kiwi = Kiwi()
def kiwi_tokenize(text: str):
    return [t.form for t in _kiwi.tokenize(text)]

bm25_ws = BM25Retriever.from_documents(chunk_docs, preprocess_func=ws_tokenize)
bm25_kiwi = BM25Retriever.from_documents(chunk_docs, preprocess_func=kiwi_tokenize)

def bm25_search(query: str, k: int, tokenizer: str = "kiwi"):
    r = bm25_kiwi if tokenizer == "kiwi" else bm25_ws
    r.k = k
    return r.invoke(query)

# (3) RRF 융합
def _doc_id(d):
    m = d.metadata
    return (m.get("filename"), m.get("page"), d.page_content[:80])

def rrf_fuse(result_lists, top_k: int, k: int = RRF_K):
    scores, docmap = {}, {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = _doc_id(doc)
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank + 1)
            docmap[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)[:top_k]
    return [docmap[key] for key in ranked]

# (4) Re-ranker (BGE cross-encoder)
from sentence_transformers import CrossEncoder
_reranker = CrossEncoder(RERANKER_MODEL, device=EMBED_DEVICE)

def rerank(query: str, docs, top_n: int):
    if not docs:
        return docs
    pairs = [(query, d.page_content) for d in docs]
    scores = _reranker.predict(pairs)
    order = sorted(range(len(docs)), key=lambda i: scores[i], reverse=True)
    return [docs[i] for i in order[:top_n]]

# (5) Multi-Query (qwen3:4b로 변형 질의 생성)
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
_mq_llm = ChatOllama(model=OLLAMA_MODEL, temperature=0.3)

def gen_queries(query: str, n: int = MULTI_QUERY_N):
    prompt = (f"다음 질문을 의미는 유지하되 검색에 유리하도록 서로 다른 표현으로 {n}개 재작성하세요. "
              f"동의어/유사 의학용어를 활용하고, 한 줄에 하나씩, 번호나 설명 없이 질문만 출력하세요.\n\n질문: {query}")
    out = (_mq_llm | StrOutputParser()).invoke(prompt)
    variants = [l.strip(" -0123456789.") for l in out.split("\n") if l.strip()]
    variants = [v for v in variants if v and v != query][:n]
    return [query] + variants
print("retriever 구성 요소 준비 완료")

---
## 4. config별 검색 디스패처

In [ ]:
def retrieve(query: str, config: str):
    if config == "dense_only":
        return dense_search(query, TOP_K)
    if config == "bm25_ws":
        return bm25_search(query, TOP_K, tokenizer="ws")
    if config == "bm25_kiwi":
        return bm25_search(query, TOP_K, tokenizer="kiwi")
    if config == "hybrid":
        d = dense_search(query, RRF_FETCH)
        b = bm25_search(query, RRF_FETCH, tokenizer="kiwi")
        return rrf_fuse([d, b], top_k=TOP_K)
    if config == "hybrid_rerank":
        d = dense_search(query, RERANK_FETCH)
        b = bm25_search(query, RERANK_FETCH, tokenizer="kiwi")
        pool = rrf_fuse([d, b], top_k=RERANK_FETCH)
        return rerank(query, pool, top_n=TOP_K)
    if config == "hybrid_rerank_mq":
        pool, seen = [], set()
        for q in gen_queries(query):
            d = dense_search(q, RRF_FETCH)
            b = bm25_search(q, RRF_FETCH, tokenizer="kiwi")
            for doc in rrf_fuse([d, b], top_k=RRF_FETCH):
                key = _doc_id(doc)
                if key not in seen:
                    seen.add(key); pool.append(doc)
        return rerank(query, pool, top_n=TOP_K)
    raise ValueError(config)

# sanity check
test_q = "HER2 양성 유방암은 어떻게 치료하나요?"
for cfg in ["dense_only", "bm25_kiwi", "hybrid_rerank"]:
    hits = retrieve(test_q, cfg)
    print(f"[{cfg}] {len(hits)}개 -> {hits[0].metadata.get('org')} p.{hits[0].metadata.get('page','?')}")

---
## 5. RAG 체인 (3·4주차와 동일 프롬프트)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOllama(model=OLLAMA_MODEL, temperature=0.0)

RAG_PROMPT = ChatPromptTemplate.from_template("""당신은 유방암 정보 검색 보조 시스템입니다.

아래 [참고 문서]만 사용해서 [질문]에 답변하세요. 문서에 없는 내용은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다"라고 답하세요.
답변 마지막에는 반드시 다음 두 가지를 포함하세요:
1. 출처: 참고한 문서명과 페이지 (예: 출처: 국립암센터 유방암 검진 권고안, p.5)
2. 면책 문구: "이 답변은 일반 정보 제공 목적이며, 실제 진단·치료는 반드시 의료진과 상의하세요."

[참고 문서]
{context}

[질문]
{question}

[답변]""")

def format_context(docs):
    parts = []
    for i, d in enumerate(docs, 1):
        m = d.metadata
        parts.append(f"[{i}] 출처: {m.get('org','?')} / {m.get('title','?')} / p.{m.get('page','?')}\n{d.page_content}")
    return "\n\n---\n\n".join(parts)

def ask(question: str, config: str):
    docs = retrieve(question, config)
    prompt = RAG_PROMPT.format(context=format_context(docs), question=question)
    answer = (llm | StrOutputParser()).invoke(prompt)
    return {"question": question, "answer": answer,
            "contexts": [d.page_content for d in docs]}

---
## 6. 평가 질문 세트 (golden_set_v0 재사용)

In [ ]:
import pandas as pd

golden_path = DATA_EVAL / "golden_set_v0.csv"
df_golden = pd.read_csv(golden_path)
golden = df_golden.to_dict("records")
print(f"golden_set_v0: {len(golden)}문항")
df_golden[["question"]]

---
## 7. config별 실행 + RAGAS + latency

주의: config x 10문항 생성 + RAGAS. qwen3:4b CPU 기준 config 1개당 수십 분.
- `RUN_CONFIGS`를 줄여 단계적으로 돌리는 것을 권장 (먼저 dense_only/bm25 2종/hybrid → 그다음 rerank/mq)
- `dense_only`는 `REUSE_WEEK4_DENSE=True`면 4주차 전략 C 결과를 그대로 사용

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from tqdm import tqdm

ragas_llm = LangchainLLMWrapper(ChatOllama(model=OLLAMA_MODEL, temperature=0.0))
ragas_emb = LangchainEmbeddingsWrapper(embeddings)
METRICS = [faithfulness, answer_relevancy, context_precision]
METRIC_COLS = ["faithfulness", "answer_relevancy", "context_precision"]

def run_config(config: str):
    rows, lat = [], []
    for g in tqdm(golden, desc=f"RAG[{config}]"):
        t0 = time.time()
        r = ask(g["question"], config)
        lat.append(time.time() - t0)
        rows.append({"question": r["question"], "answer": r["answer"],
                     "contexts": r["contexts"], "ground_truth": g.get("ground_truth", "")})
    ds = Dataset.from_list(rows)
    scores = evaluate(dataset=ds, metrics=METRICS, llm=ragas_llm, embeddings=ragas_emb)
    df = scores.to_pandas()
    df["latency_s"] = lat
    df.to_csv(DATA_PROCESSED / f"week5_ragas_{config}.csv", index=False, encoding="utf-8-sig")
    return df

score_tables = {}
for cfg in RUN_CONFIGS:
    if cfg == "dense_only" and REUSE_WEEK4_DENSE:
        w4 = DATA_PROCESSED / f"week4_ragas_{CHOSEN_CHUNK_STRATEGY}.csv"
        if w4.exists():
            score_tables[cfg] = pd.read_csv(w4)
            print(f"{cfg}: 4주차 전략 {CHOSEN_CHUNK_STRATEGY} 결과 재사용")
            continue
        print(f"{cfg}: 4주차 결과 없음 -> 직접 실행")
    score_tables[cfg] = run_config(cfg)
    print(f"{cfg}: 완료")

---
## 8. Ablation 결과표 (baseline 대비 개선폭)

In [ ]:
rows = []
for cfg in RUN_CONFIGS:
    df = score_tables[cfg]
    row = {"config": cfg}
    for col in METRIC_COLS:
        row[col] = round(df[col].mean(), 4) if col in df.columns else None
    row["avg_latency_s"] = round(df["latency_s"].mean(), 2) if "latency_s" in df.columns else None
    rows.append(row)

df_ab = pd.DataFrame(rows)
if "dense_only" in set(df_ab["config"]):
    base = df_ab[df_ab["config"] == "dense_only"].iloc[0]
    for col in METRIC_COLS:
        df_ab[col + "_delta"] = (df_ab[col] - base[col]).round(4)

df_ab.to_csv(DATA_PROCESSED / "week5_retrieval_ablation.csv", index=False, encoding="utf-8-sig")
print("저장: week5_retrieval_ablation.csv")
df_ab

---
## 9. Error case 분석 (검색 실패 3개 이상)

최종 채택 후보(`hybrid_rerank` 또는 `hybrid_rerank_mq`)에서 context_precision이 낮은 질문을 뽑아, 왜 검색이 실패했는지 원인을 적는다. (ADR / retrospective 근거)

In [ ]:
TARGET = "hybrid_rerank_mq" if "hybrid_rerank_mq" in score_tables else "hybrid_rerank"
df = score_tables.get(TARGET)
if df is not None and "context_precision" in df.columns:
    worst = df.nsmallest(3, "context_precision")
    for _, r in worst.iterrows():
        print("=" * 64)
        print(f"질문: {r['question']}")
        print(f"context_precision={r['context_precision']:.3f}  faithfulness={r.get('faithfulness', float('nan')):.3f}")
        ctxs = r["contexts"] if isinstance(r["contexts"], list) else []
        for i, c in enumerate(ctxs[:3], 1):
            print(f"  [{i}] {str(c)[:160].replace(chr(10),' ')} ...")
        print("  원인 가설(직접 작성): ")
        print()

---
## 10. ADR / retrospective 추가용 메모

`docs/adr/` 또는 `docs/week5_retrospective.md`에 옮긴다.

### 결과 해석 가이드
| 비교 | 보는 것 |
|---|---|
| bm25_ws vs bm25_kiwi | **한국어 토크나이저 효과**. Kiwi가 context_precision을 올렸다면 조사/어미 분리가 sparse 매칭을 개선한 것 |
| dense_only vs hybrid | sparse가 dense의 약점(정확한 용어·약물명·"HER2" 같은 키워드)을 보완했는가 |
| hybrid vs hybrid_rerank | cross-encoder 재정렬이 상위 chunk 정확도를 높였는가 (latency 증가 대비 가치) |
| hybrid_rerank vs +mq | query 변환이 recall을 늘렸는가, 아니면 노이즈만 늘렸는가 |

### ADR에 적을 것 (왜 이 retrieval 전략을 선택했는가)
- 채택한 최종 구성과 그 근거 (수치 기반)
- 트레이드오프: rerank/MQ는 latency가 N배 증가 (8번 표의 avg_latency_s 인용) → 운영 관점에서 어디까지 둘지
- 한국어 BM25 토크나이저 결정 근거 (Kiwi 채택/미채택)
- 데이터 진단 연결: 표 많은 임상 가이드라인(대한유방암학회)에서 rerank가 특히 효과 본 사례가 있었는지

### 5주차 예고로 남겼던 Self-Query Retriever
이번엔 ablation에서 제외. `language`/`org`/`doc_type` 메타 기반 필터링은 6주차 Agentic RAG의 라우팅 로직과 함께 다루면 자연스럽다 (검색 전 메타 필터 → Agentic 재검색 루프).

### 면접 한 줄
> "dense 단독에서 한국어 정확 용어 매칭이 약해 Hybrid(BM25+Dense, RRF)를 넣고, 표 중심 임상 가이드라인 검색을 위해 BGE cross-encoder 재정렬을 추가했습니다. BM25 토크나이저는 공백 분리 대비 Kiwi 형태소가 context precision을 X% 올려 채택했습니다. 단, rerank+MQ는 latency가 N배라 운영 기준으로 hybrid_rerank를 기본값으로 두었습니다."
